# Introduzione a PyTorch Geometric (PyG)
Benvenuto a PyTorch Geometric! Dal momento che conosci già PyTorch, la curva di apprendimento sarà veloce. PyG non fa altro che estendere i blocchi costruttivi fondamentali di PyTorch `torch.nn.Module`, aggiungendo la gestione delle matrici sparse (per gli archi) e moduli per il Message Passing Graph Neural Networks (MPNN).

In [1]:
# Import delle librerie base e funzioni PyG
import torch
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
import networkx as nx

# Per riprodurre l'esperimento fedelmente
torch.manual_seed(42)

/home/claudio-fantasia/miniconda3/envs/geometricDL/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Come definire un grafo: Il PyG `Data` Object
A differenza di NetworkX, PyTorch Geometric memorizza l'informazione *in modo sparso* a livello scalare. Il blocco base è l'oggetto `torch_geometric.data.Data`.
I componenti principali sono:
- **`x`**: Tensor `[num_nodes, num_node_features]`, che rappresenta tutte le feature scalari di ogni nodo in formato listato.
- **`edge_index`**: Tensor `[2, num_edges]`, che indica i collegamenti tra i nodi nel formato COO (Coordinate Format, ovvero un array di [Nodi Sorgente] combinato a un array di [Nodi Destinazione]).
- **`y`**: Tensor (generalmente per task specifici, etichetta dell'intero grafo, edge, o dei nodi).

In [10]:
# Definiamo un mini-grafo di 3 nodi con connessioni (1 -> 0), e (1 -> 2).
# Considera che se il grafo non é diretto dobbiamo aggiungere (0 -> 1) e (2 -> 1) manualmente.
edge_index = torch.tensor([[1, 1, 0, 2],
                           [0, 2, 1, 1]], dtype=torch.long) # Formato COO [1->0, 1->2, 0->1, 2->1]

# Ora diamo a ciascun nodo una "feature" fissa (in questo caso numeri per i 3 nodi)
x = torch.tensor([[-1], [0], [1]], dtype=torch.float)

# Creiamo l'oggetto `Data` fornito da PyTorch Geometric
data = Data(x=x, edge_index=edge_index)

print(f'Il numero totale dei nodi é {data.num_nodes}')
print(f'Il numero totale degli archi (bidirezionali considerati distinti) {data.num_edges}')
print(f'Il parametro y e presente? {"Si" if "y" in data else "No"}')
print(f'\nStruttura PyTorch:\n{data}')

# Convertiamo per testare il passaggio inverso.
nx_graph = nx.to_networkx_graph(data)
print(f'In NetworkX: nodes={nx_graph.nodes}, edges={nx_graph.edges}')

Il numero totale dei nodi é 3
Il numero totale degli archi (bidirezionali considerati distinti) 4
Il parametro y e presente? No

Struttura PyTorch:
Data(x=[3, 1], edge_index=[2, 4])
In NetworkX: nodes=['x', tensor([[-1.],
        [ 0.],
        [ 1.]]), 'edge_index', tensor([[1, 1, 0, 2],
        [0, 2, 1, 1]])], edges=[('x', tensor([[-1.],
        [ 0.],
        [ 1.]])), ('edge_index', tensor([[1, 1, 0, 2],
        [0, 2, 1, 1]]))]


In [13]:
print(edge_index[:,2])

tensor([0, 1])


## 2. Da NetworkX a PyG
Tu stai calcolando tutti i calcoli di Gromov su grafi definiti da NetworkX in `src/graphs/utils.py`.
Puoi facilmente usare la libreria di compatibilità `from_networkx` e trasformare la topologia rewired in un tensore. L'unica cosa che devi ricordare è assegnare le feature desiderate (PyG non le estrae implicitamente).

In [ ]:
# Prendiamo un grafo generato dai tuoi utils (e.g. uno Star Graph o un Cycle)
# (Assicurati di lanciare python dal root o di aver configurato sys.path per importare src.graphs)
import sys; sys.path.append('../')
from src.graphs.utils import create_graph

# Usiamo un Erdos Renyi come proxy
g_nx = create_graph(type='erdos_renyi', n=10, p=0.3)

# ORA diamogli features artificiali (1 per ogni nodo) ed etichette puramente per l'esempio (2 classi)
# Queste non verrebbero importate di default, quindi le dobbiamo includere nella metadati "node",
# cosí `from_networkx` se le prende automaticamente.
for node in g_nx.nodes():
    g_nx.nodes[node]['x'] = [float(1)]   # In PyG vuole un List/Tensor N x Channels
    g_nx.nodes[node]['y'] = node % 2     # Etichette di classe (es. nodo pari/nodi dispari)

pyg_data = from_networkx(g_nx)

print(f"Proprietá PyG da NetworkX riconvertito:\n")
print(f"Edge Index shape: {pyg_data.edge_index.shape}")
print(f"Features: {pyg_data.x}")
print(f"Etichette associate: {pyg_data.y}")

## 3. Dataset per esplorare: KarateClub
PyTorch Geometric fornisce diversi toy-dataset. Useremo il **KarateClub**.
Oltre a questo, puoi trovare facilmente tutti i dataset LRGB per testare l'oversquashing.

In [14]:
from torch_geometric.datasets import KarateClub

dataset = KarateClub()
print(f'Numero di grafi nel dataset: {len(dataset)}')
print(f'Numero di feature per nodo: {dataset.num_features}')
print(f'Numero di classi (4 fazioni): {dataset.num_classes}')

# Otteniamo il primo (e unico) grafo del KarateClub
data = dataset[0]  

# Il dataset include una `train_mask` (un vettore di True/False che 
# ti dice per ciascun nodo della label se puó usarla per il training, o valid/test)
# e una metrica y con la label per ogni classe (0, 1, 2, o 3)
print(data)

Numero di grafi nel dataset: 1
Numero di feature per nodo: 34
Numero di classi (4 fazioni): 4
Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])


In [22]:
data['train_mask'] 

tensor([ True, False, False, False,  True, False, False, False,  True, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False,  True, False, False, False, False, False,
        False, False, False, False])

## 4. Costruiamo e alleniamo una rete (GCN) classica
Sei giá abituato con PyTorch `nn.Module`. Aggiungere la convoluzione sui branchi richiede chiamare uno dei layer, che puoi importare da `torch_geometric.nn` (es. `GCNConv`).
Ne richiamiamo uno gia' costruito nel file `src/models/baselines.py` o lo ridefiniamo qui.

In [23]:
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        # Inizializziamo i nostri blocchi GCN (Graph Convolutions).
        # Sono parenti equivalenti a Linear o Conv2di normalissimi moduli PyTorch.
        # Dataset è il nostro `dataset` istituito sopra.
        self.conv1 = GCNConv(dataset.num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index):
        # Passo 1: Eseguiamo Convoluzione sui Nodi (Messaggistica con i vicini del target, o 1-hop neighborhood)
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Passo 2: Nuova iterazione di Convoluzione sui nodi (Messaggi a 2-hop di distanza)
        x = self.conv2(x, edge_index)
        return x

model = GCN(hidden_channels=16)
print(model)

GCN(
  (conv1): GCNConv(34, 16)
  (conv2): GCNConv(16, 4)
)


In [25]:
model.training

True

## 5. Allenare il Modello
Tutto cio' si riduce al classico loop Pytorch. Fai una `.forward()` sui dati del grafo, calcoli la loss sui nodi annotati (nel nostro caso i nodi "trainabili" `data.train_mask`), propaghi indietro gli errori.

In [26]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4) # Adam Optimizer standard
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()               # Setta la modalitá a Training (e.g. Dropout attivi)
    optimizer.zero_grad()       # Azzera gradienti (se non metti questo a Pytorch gli dispiace)
    out = model(data.x, data.edge_index)  # Calcolo del Forward Pass. Passiamo `x` e `edge_index`.
    
    # Calcoliamo la loss SOLO sui nodi del training loop usando il "masking"
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    
    loss.backward()             # Derivazione parziale della Loss (Autograd).
    optimizer.step()            # Aggiorna il Peso Nascosto della Rete.
    return loss

for epoch in range(1, 201):     # Training molto veloce di 200 epochs
    loss = train()
    if epoch % 25 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

print("\n Addestramento Riuscito! 🚀")

Epoch: 025, Loss: 0.9334
Epoch: 050, Loss: 0.4093
Epoch: 075, Loss: 0.2151
Epoch: 100, Loss: 0.0727
Epoch: 125, Loss: 0.0317
Epoch: 150, Loss: 0.0369
Epoch: 175, Loss: 0.0398
Epoch: 200, Loss: 0.0284

 Addestramento Riuscito! 🚀


In [27]:
test_out = model(data.x, data.edge_index)

In [30]:
print(test_out.size())

torch.Size([34, 4])


In [ ]:
# 1. Valutazione sui nodi NON di training
# Passiamo in modalità valutazione per disattivare layer come il Dropout
model.eval()

# Ricalcoliamo l'output
out = model(data.x, data.edge_index)

# La predizione è la classe con l'attivazione più alta
pred = out.argmax(dim=1)

# Usiamo l'operatore logico NOT (~) per prendere tutti i nodi che NON erano nel train set
test_mask = ~data.train_mask
test_correct = pred[test_mask] == data.y[test_mask]

# Calcoliamo l'accuratezza
test_acc = int(test_correct.sum()) / int(test_mask.sum())
print(f'Accuracy sui nodi che non hanno partecipato al train: {test_acc:.4f}')

RuntimeError: Boolean value of Tensor with more than one value is ambiguous

## 6. Lavorare con Multipli Grafi: DataLoader di PyG e Mini-Batching
Spesso non avrai un singolo mega-grafo (come i social network o il club di Karate), ma tantissimi grafi indipendenti di cui vuoi predire una proprietà (es. chimica computazionale con molecole).

PyTorch Geometric gestisce l'ottimizzazione raggruppando più piccoli grafi unendoli in un unico "grafo disconnesso" e gigante nel batch corrente. Per fare questo si usa il `DataLoader`.

La funzione `forward()` lavorerà automaticamente con la grandezza del layer raggruppato. Alla fine si usa una tecnica di `pooling` (leggendo il campo speciale `batch`).

In [37]:
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool # Operazione di Readout (Aggregation)

# 1. Carichiamo un Dataset con una grandissima quantità di grafi
# MUTAG è un classico dataset di 188 grafi rappresentanti composti chimici
dataset_multi = TUDataset(root='/tmp/MUTAG', name='MUTAG')

print(f"Grafi nel dataset: {len(dataset_multi)}")
print(f"Num classi per l'intero grafo: {dataset_multi.num_classes}\n")


Grafi nel dataset: 188
Num classi per l'intero grafo: 2



Processing...
Done!


In [ ]:
# 2. Suddividiamo Randomicamente Train e Test
dataset_multi = dataset_multi.shuffle()
train_dataset = dataset_multi[:150]
test_dataset = dataset_multi[150:]

# 3. Creiamo un DataLoader (Pytorch Geometric batcher)
# I batch contengono grafi raggruppati in un grande unico tensore
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 4. Spaziatura su un modello dummy (Solo per dimostrare il passaggio dei Parametri del Batch)
for step, batch in enumerate(train_loader):
    print(f'Batch {step + 1}:')
    print('==========================\n')
    print(f'Dimensioni del Mini-batch aggregato (Nodi tot, Archi tot): {batch}')
    
    # Questo tensore `batch` aiuta il modello a ricordarsi a quale input originale 
    # di grafo apparteneva ogni nodo dopo l'aggregazione matriciale `[0, 0, 0, 1, 1...]`
    print(f'\nA quale grafo (0-{batch.num_graphs - 1}) il nodo appartiene:\n{batch.batch[:15]}...\n')
    
    break # Fermiamo al primo batch


Batch 1:

Dimensioni del Mini-batch aggregato (Nodi tot, Archi tot): DataBatch(edge_index=[2, 1384], x=[619, 7], edge_attr=[1384, 4], y=[32], batch=[619], ptr=[33])

A quale grafo (0-31) il nodo appartiene:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])...



NameError: name 'self' is not defined

## 7. Rete Graph Convolutional per Classificazione di Itero Grafo
A differenza del Karaoke Club dove l'output era sul singolo nodo scendendo le feature al numero di classi desiderato `(N, classes)`, qui l'obiettivo è processare i *Message Passing* dei nodi `(N, hidden)`, ed infine riassumerli/aggregare gli hint di un intero minigrafo in una sola riga tramite la funzione di *Readout* (es. `global_mean_pool`), per avere un tensore finale `(Batch_size, classes)`.

In [52]:
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

class GCNGraph(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCNGraph, self).__init__()
        torch.manual_seed(12345)
        # Layer convoluzionali sui nodi (Message Passing)
        # 3 Hop neighborhood
        self.conv1 = GCNConv(dataset_multi.num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        
        # Classifier lineare per le feature aggregate dell'intero grafo
        self.lin = Linear(hidden_channels, dataset_multi.num_classes)

    def forward(self, x, edge_index, batch):
        # 1. Otteniamo node embeddings
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)

        # 2. Readout layer: Aggrega le feature dei singoli nodi calcolandone la media globale
        # Usando il parametro passatogli in ingresso "batch" PyG capirà come raggrupparli 
        # Risultato -> [batch_size, hidden_channels]
        print(f'Prima del pooling: {x.size()}')
        x = global_mean_pool(x, batch)
        print(f'Dopo il pooling: {x.size()}')

        # 3. Applichiamo uno strato Fully Connected finale (Classificatore Lineare finale)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        
        return x

model_graph = GCNGraph(hidden_channels=64)
print(model_graph)

GCNGraph(
  (conv1): GCNConv(7, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 64)
  (lin): Linear(in_features=64, out_features=2, bias=True)
)


## 8. Ciclo di addestramento su grafi multipli
Rispetto all'esempio base di Node Classification cambiano poche cose:
1. Devi iterare sul generatori in batch creati dal `DataLoader`.
2. Ognuno di questi iterabili fornisce l'oggetto `batch` (che ha dentro `.x`, `.edge_index`, `.y` e `.batch`).
3. Non hai più una singola maschera che indica dove calcolare la loss (`train_mask`), qua la loss (`y`) si calcola alla fine della predizione sul mini-gruppo direttamente.
4. Passiamo il parametro `.batch` alla `forward()` del nostro nuovo modello.

In [53]:
optimizer_g = torch.optim.Adam(model_graph.parameters(), lr=0.01)
criterion_g = torch.nn.CrossEntropyLoss()

def train_graphs():
    model_graph.train() # Imposta modalo su Training
    total_loss = 0      # Sommo per calcolare la media alla fine del batch
    
    # 1. Al posto del Forward su tutto il dataset... Itero i vari Minibatch raggruppabili (es. 32 alla volta)
    for data in train_loader:
        optimizer_g.zero_grad()  
        
        # 2. Richiamo il Forward su `data.x`, `data.edge_index`, ma noto: Devo passargli il "bookkeeper" (.batch)
        out = model_graph(data.x, data.edge_index, data.batch)  
        
        # 3. La Loss è semplice: Predizione (batch_size vs classi) al vero batch target `.y` (N x classes - nessun masking).
        loss = criterion_g(out, data.y)  
        loss.backward()  
        optimizer_g.step()  
        total_loss += loss.item() * data.num_graphs # Scalo dinamicamente se l'ultimo batch ha meno di 32 elementi

    return total_loss / len(train_loader.dataset) # Calcolo della media batch in tutto il set

def test_graphs(loader):
    model_graph.eval() # Imposto il modulo Test
    correct = 0        # Conteggi di quelle vere predirrate in tutto il Set
    
    for data in loader:  # Iteriamo sui batch (Train _oppure_ Test)
        out = model_graph(data.x, data.edge_index, data.batch)  
        pred = out.argmax(dim=1)  # La classe predetta è quella con l'output max value (logistica soft-max-like)
        correct += int((pred == data.y).sum())  
    
    return correct / len(loader.dataset)  # Ritorno Accuratezza totale calcolata su tutti i grafi (150/150 - o 38/38)

# Eseguiamo il Training: 
print("Inizio Addestramento su MUTAG...\n")
for epoch in range(1, 101):
    loss = train_graphs()
    
    # Valutiamo l'Accuratezza in Tempo Reale
    train_acc = test_graphs(train_loader)
    test_acc = test_graphs(test_loader)
    
    if epoch % 10 == 0: # Clicka sulla stampa ogni 10 Epoch
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')

print("\n🚀 Finito! Guarda le performances del Mini Batching!")

Inizio Addestramento su MUTAG...

Prima del pooling: torch.Size([573, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([556, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([567, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([586, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([412, 64])
Dopo il pooling: torch.Size([22, 64])
Prima del pooling: torch.Size([572, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([578, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([538, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([561, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([445, 64])
Dopo il pooling: torch.Size([22, 64])
Prima del pooling: torch.Size([574, 64])
Dopo il pooling: torch.Size([32, 64])
Prima del pooling: torch.Size([103, 64])
Dopo il pooling: torch.Size([6, 64])
Prima del pooling: 